# Adaptive functional GNN — FIXED architecture — Kaggle T4

Nested LOSO on the frozen 794-subject / 17-site dual cohort.

## The fixed model (these are now the script defaults)

| | setting | why |
|---|---|---|
| node features | **`fcrow`** — each node's full `fc_z` row (111 dims) | the 4-number `robust4` summary was an information bottleneck |
| readout | **`concat(mean, max)`** | mean-pool alone dilutes localised signal by ~1/111 |

Measured A/B on the 3 largest sites (k=20, 3 seeds):

| arm | ROC | pooled_sd |
|---|---|---|
| robust4 + mean (original) | 0.550 | 0.357 |
| robust4 + mean+max | 0.579 | 0.557 |
| **fcrow + mean+max (this model)** | **0.663** | 0.707 |

⚠️ **Do not compare that 0.663 to the 17-site 0.658 ceiling** — those three sites are
easier than average. On the *same* three sites the anchors are: robust4 flat SVM
**0.699**, edge-SVM 0b **0.756**. The fixed GNN is still below both; the fixes closed
about half the gap. The point of this run is to get a **17-site** number that *is*
comparable to 0.630 / 0.658.

## Before you run — notebook settings

1. **Accelerator → GPU T4 ×2**
2. **Internet → On**
3. **Add Data →** your private dataset with `cohort_final.csv` + `fc_z.npy` (~39 MB)

## Runtime on T4 (`fcrow` is ~4× slower per fit than `robust4`)

| Run | fits | ~time |
|---|---|---|
| **Main run** (fixed k=20, no inner search, 5 seeds) | 85 | **~3 h** ← start here |
| Optional Stage A (k sweep, 4 configs) | +204 | ~7 h more |
| Optional Stage B (capacity, 6 configs) | +306 | ~11 h more |

The **main run** is what you need for the headline number and fits comfortably in one
12 h session. The sweeps are refinement — run them in later sessions if you want them.

**Checkpointing is per `(site, seed)`**: every finished fold-seed is appended to the
CSV immediately, and the inner-HP choice is cached per site. A killed session loses at
most one fold. See the last cell for the resume procedure.

In [ ]:
# 1. Confirm the GPU
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Get the code
!rm -rf /kaggle/working/Thesis-with-ABIDE-dataset
!git clone -q https://github.com/hazrotali2003028/Thesis-with-ABIDE-dataset.git /kaggle/working/Thesis-with-ABIDE-dataset
%cd /kaggle/working/Thesis-with-ABIDE-dataset
!git log --oneline -1

In [ ]:
# 3. Dependencies. torch + CUDA are preinstalled on Kaggle.
#    bctpy is needed for robust4; nolds is imported at module level in
#    node_features.py, so install it too even though robust4 does not use it.
!pip -q install torch_geometric bctpy nolds
import numpy, torch
print('numpy', numpy.__version__, '| torch still OK:', torch.cuda.is_available())

In [ ]:
# 4. Stage the uploaded data into the layout the script expects:
#      <data-dir>/cohort_final.csv  and  <data-dir>/cache/fc_z.npy
import glob, os, shutil

DATA = '/kaggle/working/data'
os.makedirs(os.path.join(DATA, 'cache'), exist_ok=True)

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input — did you attach the dataset?'
    return hits[0]

shutil.copy(find('cohort_final.csv'), f'{DATA}/cohort_final.csv')
shutil.copy(find('fc_z.npy'), f'{DATA}/cache/fc_z.npy')

import pandas as pd, numpy as np
print('cohort', pd.read_csv(f'{DATA}/cohort_final.csv').shape,
      '| fc_z', np.load(f'{DATA}/cache/fc_z.npy').shape)

In [ ]:
# 5. RESUME (no-op on a first run).
#    If you attached a previous run's notebook output, restore its result CSV and
#    inner-HP cache so finished (site, seed) rows are skipped.
import glob, shutil, os
restored = 0
for pat in ('adaptive_gnn_stage*.csv', 'adaptive_gnn_stage*_hp.json'):
    for f in glob.glob(f'/kaggle/input/**/{pat}', recursive=True):
        shutil.copy(f, '/kaggle/working/'); restored += 1
        print('restored', os.path.basename(f))
print('nothing to resume — fresh run' if not restored else f'{restored} file(s) restored')

## Main run — the fixed GNN, all 17 sites

`fcrow` node features + `mean+max` readout (script defaults), `k=20` (the value
Stage A's honest inner CV selected most often: 43/87), `layers=2, hidden=64`,
5 seeds, 60 epochs.

`--skip-inner` uses that fixed config rather than re-searching per site. This is the
deliberate trade: the inner k-sweep costs ~7 h extra on `fcrow`, and k=20 was already
selected honestly on training sites only in the earlier run. State this in the writeup
as "k fixed at the value selected by the prior nested search" — it is a reused honest
choice, not a peek at test data.

Smoke-test first (2 sites, 1 seed, 6 epochs, ~1 min): it confirms the data mounted and
that the **adaptive-topology unit test passes** (`std(edge_index) > 0`). If that assert
fires, the adaptive relation is vacuous and the whole run is meaningless — better to
find out in 60 seconds than 3 hours.

In [ ]:
!python dualgraph/train_adaptive_gnn.py --stage a --smoke \
    --data-dir /kaggle/working/data --out-dir /kaggle/working

In [ ]:
# MAIN RUN — fixed GNN, 17 sites, 5 seeds (~3 h on T4).
# Re-runnable: finished (site, seed) rows are skipped on resume.
!python dualgraph/train_adaptive_gnn.py \
    --stage b --best-k 20 --skip-inner \
    --node-mode fcrow --readout meanmax \
    --seeds 5 --epochs 60 --tag _fixed \
    --data-dir /kaggle/working/data --out-dir /kaggle/working

In [ ]:
# Headline result of the main run, against the like-for-like anchors.
import pandas as pd, sys
sys.path.insert(0, '/kaggle/working/Thesis-with-ABIDE-dataset/dualgraph')
from train_adaptive_gnn import report

d = pd.read_csv('/kaggle/working/adaptive_gnn_stageb_fixed.csv')
print(f"rows {len(d)} / expected 85   sites {d.site.nunique()} / 17")
report(d)   # honest table + paper1 table + per-metric inflation

## OPTIONAL sweeps — only if you have session time left

These are refinement, not required for the headline number. Both are expensive on
`fcrow` (~4× the per-fit cost of `robust4`), so treat each as its own session.

**Stage A** — re-sweep `k ∈ {5,10,20,30}` for the fixed architecture (~7 h). Worth it
only if you want k re-selected under `fcrow` rather than inherited from the `robust4`
run.

**Stage B** — sweep `layers ∈ {2,3,4} × hidden ∈ {64,128}` (~11 h, needs its own
session). The thing to watch is **`pooled_sd` by depth**: at k=20 two hops already
reach the whole 111-node graph, so layers 3–4 may only re-mix and collapse the pooled
vector. If `pooled_sd` drops sharply at `layers=4`, that is oversmoothing measured
directly — far stronger evidence than a 0.02 AUC difference across 17 noisy sites where
the MDE is 0.06.

In [ ]:
# OPTIONAL Stage A — k sweep under fcrow (~7 h). Uncomment to run.
# !python dualgraph/train_adaptive_gnn.py --stage a \
#     --node-mode fcrow --readout meanmax --seeds 5 --epochs 60 --tag _fcrow \
#     --data-dir /kaggle/working/data --out-dir /kaggle/working

# OPTIONAL Stage B — capacity sweep (~11 h, own session). Set BEST_K first.
# !python dualgraph/train_adaptive_gnn.py --stage b --best-k 20 \
#     --node-mode fcrow --readout meanmax --seeds 5 --epochs 60 --tag _cap \
#     --data-dir /kaggle/working/data --out-dir /kaggle/working
print('optional sweeps are commented out — run the main cell above first')

## Results

The **honest** columns are the reportable result. The `paper1_*` columns use
Paper 1's epoch-selection rule (epoch chosen on the test site) and are for
comparison only — measured on this dataset that rule inflates AUC by **+0.100
(p = 1.9e-06)**, so it reads the answer key and must never be reported as a result.
`inflation = paper1 − honest` quantifies the gap per site.

In [ ]:
import sys, glob, pandas as pd
sys.path.insert(0, '/kaggle/working/Thesis-with-ABIDE-dataset/dualgraph')
from train_adaptive_gnn import report

# report() prints, for each stage: the HONEST per-site table, the PAPER1 per-site
# table (same metric set), and the per-metric inflation (paper1 - honest).
for f in sorted(glob.glob('/kaggle/working/adaptive_gnn_stage*.csv')):
    print('=' * 78); print(f)
    report(pd.read_csv(f))

print('\nanchors:  edge-SVM 0b = 0.658 | robust4 flat SVM = 0.630 | motion floor = 0.561')
print('the question is not "beat 0.658" (both channels derive from fc_z, so that is')
print('the ceiling) but "does message passing beat 0.630, the same features flat?"')

In [ ]:
# Per-site detail + checkpoint status of the main run
import pandas as pd, os, json
p = '/kaggle/working/adaptive_gnn_stageb_fixed.csv'
if os.path.exists(p):
    d = pd.read_csv(p)
    done = d.groupby('site').seed.nunique()
    print('CHECKPOINT STATUS (seeds completed per site, expect 5):')
    print(done.to_string())
    missing = [s for s in d.site.unique() if done[s] < 5]
    print('\nincomplete sites:', missing if missing else 'none — run finished')
    print(f'\nduplicate (site,seed) rows: {d.duplicated(["site","seed"]).sum()}')
    print('\nper-site honest ROC vs same-site anchors:')
    print(d.groupby('site')[['honest_roc_auc','paper1_roc_auc','inflation','pooled_sd']]
            .mean().round(3).to_string())
else:
    print('main run has not produced output yet')

---
# Stages 7–8 — dual-graph fusion and cross-attention

**Extra data required:** add `features/abide_features_raw.csv` (4.5 MB) and
`features/node_coords.csv` (8 KB) to your Kaggle dataset, or every structural and
dual rung fails at load.

| rung | model | what it tests |
|---|---|---|
| 1a | structural GNN | the structural arm alone |
| 1b | functional GNN (fcrow) | the functional arm alone |
| 1a-mlp / 1b-mlp | same encoder + readout, **message passing removed** | isolates the **edges** from the encoder |
| **2** | dual-graph concat fusion, no attention | **gate:** `2 > max(1a,1b)`? |
| 3 | cross-attention **Q=S, K=V=F** (sMRI→fMRI) | the novelty |
| 4 | cross-attention **Q=F, K=V=S** (CAS-GNN direction) | direction matters? |

Fixed from prior experiments: k=20, layers=2, hidden=64, dropout 0.3, BatchNorm,
mean+max readout, 5 seeds, 60 epochs (no early stopping). Both epoch rules and the
full 14-metric set are produced per rung, with per-`(site,seed)` checkpointing.

**Harmonisation note:** unlike the earlier GNN runs, these apply **per-fold ComBat to
both arms** (train rows only) — matching the flat SVM baselines. The earlier GNN runs
had no ComBat, so `1b` here is *not* directly comparable to the fixed-GNN result above;
the difference is a clean read on what ComBat was worth.

**Runtime ~5–8 h on T4 for all seven** — fits one session. Run 1a/1b/MLPs/2 first,
check the gate, then decide on 3/4.

In [ ]:
# Stage 7 + prerequisites (~4 h on T4). Each rung checkpoints per (site, seed).
DATA = '/kaggle/working/data'; OUT = '/kaggle/working'
for rung in ['1a', '1b', '1a-mlp', '1b-mlp', '2']:
    print('=' * 70, f'\nRUNG {rung}')
    !python dualgraph/train_dual.py --rung {rung} --seeds 5 --epochs 60 \
        --data-dir {DATA} --out-dir {OUT}

In [ ]:
# THE STAGE-7 GATE — decides whether Stage 8 is worth 4 more hours.
import pandas as pd, numpy as np, glob, os
from scipy.stats import wilcoxon

def per_site(rung):
    p = f'/kaggle/working/dual_rung{rung}.csv'
    return pd.read_csv(p).groupby('site').honest_roc_auc.mean() if os.path.exists(p) else None

r = {k: per_site(k) for k in ['1a', '1b', '1a-mlp', '1b-mlp', '2']}
print('mean honest LOSO ROC:')
for k, v in r.items():
    if v is not None:
        print(f'  {k:8} {v.mean():.4f} +/- {v.std():.4f}  (n={len(v)})')

if r['2'] is not None and r['1a'] is not None and r['1b'] is not None:
    best = '1b' if r['1b'].mean() >= r['1a'].mean() else '1a'
    a, b = r['2'], r[best]
    c = a.index.intersection(b.index)
    d = np.median(a.loc[c] - b.loc[c]); w, p = wilcoxon(a.loc[c], b.loc[c])
    print(f"\nGATE  rung2 vs max(1a,1b)={best}:  median dAUC={d:+.4f}  p={p:.4f}")
    print('  -> ' + ('CLEARS MDE — run Stage 8 (rungs 3, 4)' if d >= 0.06 else
          'BELOW MDE — fusion adds nothing. Kill criterion: report and SKIP Stage 8.'))

# the edges-vs-encoder test (plan section 7): GNN must beat its own MLP parity control
for gnn, mlp in [('1a', '1a-mlp'), ('1b', '1b-mlp')]:
    if r[gnn] is not None and r[mlp] is not None:
        c = r[gnn].index.intersection(r[mlp].index)
        d = np.median(r[gnn].loc[c] - r[mlp].loc[c])
        print(f"  {gnn} vs {mlp}: median dAUC={d:+.4f}  "
              f"({'edges help' if d >= 0.06 else 'edges add nothing'})")

In [ ]:
# Stage 8 — ONLY if the gate above cleared (~4 h on T4)
for rung in ['3', '4']:
    print('=' * 70, f'\nRUNG {rung}')
    !python dualgraph/train_dual.py --rung {rung} --seeds 5 --epochs 60 \
        --data-dir {DATA} --out-dir {OUT}

# full metric tables (honest + paper1 + inflation) for every completed rung
import sys, glob, pandas as pd
sys.path.insert(0, '/kaggle/working/Thesis-with-ABIDE-dataset/dualgraph')
from train_adaptive_gnn import report
for f in sorted(glob.glob('/kaggle/working/dual_rung*.csv')):
    print('=' * 78); print(f); report(pd.read_csv(f))

## If the session is killed before finishing

1. **Save Version** (commit) so `/kaggle/working` becomes the notebook's output.
2. In a new run: **Add Data → Notebook Output →** attach this notebook's previous output.
3. Run the cells from the top. Cell 5 restores `adaptive_gnn_stage*.csv` and the
   inner-HP cache, and the script skips every `(site, seed)` already finished —
   including the expensive inner HP search, which is cached per site.

Quota note: Kaggle allows 30 GPU-hours/week; the full search is ~3.5 h.